In [ ]:
# ==========================================
# NLGCL Training on Kaggle (All-in-One)
# ==========================================

import os
import shutil
import pandas as pd
import numpy as np
import sys
import zipfile
import urllib.request

# 1. System & Dependency Fixes
print(">>> fixing build environment and installing dependencies...")
try:
    # Fix import errors in pip/setuptools contexts
    !pip install -U pip setuptools wheel
    
    # Enforce NumPy < 2.0 FIRST to avoid build/runtime errors with RecBole/PyTorch
    !pip install "numpy<2.0"
    
    # Install main dependencies (removed pyg_lib, pinned colorama)
    !pip install -q recbole colorlog tensorboard "colorama>=0.4.6" pyyaml pandas hyperopt==0.2.5 --no-build-isolation
except Exception as e:
    print(f"Warning: Dependency install had issues: {e}. Continuing...")

# 2. Install PyTorch Geometric
import torch
TORCH_version = torch.__version__
CUDA_version = torch.version.cuda
print(f"Torch: {TORCH_version}, CUDA: {CUDA_version}")

# Install PyG
!pip install -q torch-geometric
!pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv

# 3. Clone Repository
if not os.path.exists("NLGCL"):
    print(">>> Cloning Repository...")
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL
else:
    if os.getcwd().split(os.sep)[-1] != "NLGCL":
        %cd NLGCL

# 4. Patch Missing Modules
print(">>> Patching Missing Modules...")
missing_modules = [
    ("lightgcn", "LightGCN"), ("hmlet", "HMLET"), ("ncl", "NCL"), ("ngcf", "NGCF"),
    ("sgl", "SGL"), ("lightgcl", "LightGCL"), ("simgcl", "SimGCL"), ("xsimgcl", "XSimGCL"),
    ("directau", "DirectAU"), ("ssl4rec", "SSL4REC"), ("dccf", "DCCF"), ("l2cl", "L2CL")
]
base_path = "recbole_gnn/model/general_recommender"
if os.path.exists(base_path):
    for module_name, class_name in missing_modules:
        file_path = os.path.join(base_path, f"{module_name}.py")
        if not os.path.exists(file_path):
            with open(file_path, 'w') as f:
                f.write(f"class {class_name}:\n    pass\n")

# 5. Dataset Preparation (Alibaba-iFashion)
# User requested 'Alibaba' dataset with specific hyperparameters.
# The Alibaba-iFashion dataset is hosted on RecBole Google Drive: https://drive.google.com/drive/folders/1so0lckI6N6_niVEYaBu-LIcpOdZf99kj
# Since it requires manual download/upload due to size/auth, we provide instructions.

print(">>> Preparing Alibaba Dataset...")
dataset_name = "alibaba"
data_dir = f"dataset/{dataset_name}"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

inter_file = os.path.join(data_dir, f"{dataset_name}.inter")

# Check if dataset exists locally or in standard Kaggle input locations
possible_locations = [
    f"/kaggle/input/{dataset_name}/{dataset_name}.inter",
    f"/kaggle/input/recbole-datasets/{dataset_name}/{dataset_name}.inter",
    f"/kaggle/input/alibaba-ifashion/{dataset_name}.inter"
]

found_dataset = False
for loc in possible_locations:
    if os.path.exists(loc):
        print(f"Found dataset at {loc}")
        shutil.copy(loc, inter_file)
        found_dataset = True
        break

if not found_dataset and not os.path.exists(inter_file):
    print("WARNING: Alibaba dataset not found automatically.")
    print("Please manually upload 'alibaba.inter' to Kaggle Input.")
    print("Download URL (RecBole GDrive): https://drive.google.com/drive/folders/1so0lckI6N6_niVEYaBu-LIcpOdZf99kj")
    
    # Fallback to ML-100K if Alibaba is missing, to allow code to run for demonstration
    print(">>> Fallback: Downloading ml-100k for demonstration...")
    fallback_name = "ml-100k"
    fallback_dir = f"dataset/{fallback_name}"
    if not os.path.exists(fallback_dir):
        os.makedirs(fallback_dir)
        
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    zip_path = os.path.join(fallback_dir, "ml-100k.zip")
    try:
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(fallback_dir)
        
        # Convert u.data to ml-100k.inter
        for root, _, files in os.walk(fallback_dir):
            if 'u.data' in files:
                u_data = os.path.join(root, 'u.data')
                df = pd.read_csv(u_data, sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
                inter_df = pd.DataFrame({
                    'user_id:token': df['user_id'],
                    'item_id:token': df['item_id'],
                    'rating:float': df['rating'],
                    'timestamp:float': df['timestamp']
                })
                inter_df.to_csv(os.path.join(fallback_dir, f"{fallback_name}.inter"), index=False, sep='\t')
                break
        dataset_name = fallback_name  # Switch to fallback
    except Exception as e:
        print(f"Fallback download failed: {e}")

# 6. Run Training
print(f">>> Starting Training with {dataset_name}...")
# Hyperparameters for Alibaba provided by user:
# n_layers=2, reg_weight=1e-4, cl_temp=0.1, alpha=0.6, cl_reg=5e-5
# If fallback to ml-100k, these might be suboptimal but code will run.
!python main.py --dataset {dataset_name} --model NLGCL --n_layers=2 --reg_weight=1e-4 --cl_temp=0.1 --alpha=0.6 --cl_reg=5e-5